In [1]:
"""
To plot rasters, for char_sp, here does good preprocessing, including pruning 
bad beh stroke (matching across SP and CHAR).

"""


'\nTo plot rasters, for char_sp, here does good preprocessing, including pruning \nbad beh stroke (matching across SP and CHAR).\n\n'

In [2]:
%load_ext autoreload
%autoreload 2

from neuralmonkey.classes.session import load_mult_session_helper
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

#%matplotlib inline

In [ ]:
# load all sessions for Pancho on 221020
animal = "Diego"
date = 231129

MS = load_mult_session_helper(date, animal)

##### Generate snippets

In [ ]:
from neuralmonkey.classes.snippets import Snippets, extraction_helper


In [ ]:
# This is best, genreeates snippest for each session, thne concats into a single SP
from neuralmonkey.classes.snippets import load_and_concat_mult_snippets
which_level = "stroke"
PRE_DUR = -0.5
POST_DUR = 0.5
EVENTS_KEEP = ["00_stroke"]
DEBUG = False
SP, _ = load_and_concat_mult_snippets(MS, which_level, EVENTS_KEEP, DEBUG = DEBUG, 
                                      prune_low_fr_sites=False, 
                                      REGENERATE_SNIPPETS=True, 
                                      PRE_DUR=PRE_DUR, POST_DUR=POST_DUR)


In [ ]:
# [OPTIONAL] Clean up, applying expt specific params and extract features
# NOTE: Only do this if you want to clean up data -- e.g, only successful trials.

from neuralmonkey.analyses.rsa import rsagood_questions_dict, rsagood_questions_params
question = "CHAR_BASE_stroke"
q_params = rsagood_questions_dict(animal, date, question)[question]

D, list_features_extraction = SP.datasetbeh_preprocess_clean_by_expt(
    ANALY_VER=q_params["ANALY_VER"], vars_extract_append=q_params["effect_vars"],
    substrokes_plot_preprocess=False)


In [ ]:
DS = SP.datasetbeh_extract_dataset('datstrokes')
sorted(DS.Dat.keys())

In [ ]:
DS.Dat["datseg"][0]

In [ ]:
sorted(list_features_extraction)

In [ ]:
# Inspect the data. 
# Each row represents a single combination of:
# (trial, chan, event). To see that, inspect the output of 

display(SP.DfScalar)

from pythonlib.tools.pandastools import grouping_print_n_samples
grouping_print_n_samples(SP.DfScalar, ["trialcode", "chan", "event_aligned"])


# Cleaning beh data

In [ ]:
# Mainly, remocing bad beh strokes, based on carefuly manual checking.

In [ ]:
DFSCALAR = SP.DfScalar.copy()

In [ ]:
# (1) shapes that are clearly bad (e.g., wrong direction)

# (2) clust_sim_max threshold.
from neuralmonkey.scripts.analy_euclidian_chars_sp import behstrokes_map_clustshape_to_thresh, params_shapes_remove


map_clustshape_to_thresh = behstrokes_map_clustshape_to_thresh(animal)
def good(x):
    sh = x["clust_sim_max_colname"]
    return x["clust_sim_max"] > map_clustshape_to_thresh[sh]
SP.DfScalar["clust_sim_max_GOOD"] = [good(row) for i, row in SP.DfScalar.iterrows()]

SP.DfScalar['clust_sim_max_GOOD'].value_counts()
SP.DfScalar = SP.DfScalar[SP.DfScalar["clust_sim_max_GOOD"]==True].reset_index(drop=True)
# Hard coded shapes to remove
shapes_remove = params_shapes_remove(animal, date, shape_var)
print("Also removing tese shapes. by hand: ", shapes_remove)
SP.DfScalar = SP.DfScalar[~SP.DfScalar[shape_var].isin(shapes_remove)]




In [ ]:
# Prune so that SP and CHAR have same shapes.
prune_version = "sp_char"
shape_var = "shape_semantic_grp"
n_min_trials_per_shape = 4
plot_counts_heatmap_savepath = None

if prune_version == "sp_char_0":
    task_kinds = ["prims_single", "character"]
    fd = {"task_kind":task_kinds, "stroke_index":[0]}
elif prune_version == "sp_char":
    task_kinds = ["prims_single", "character"]
    fd = {"task_kind":task_kinds}
elif prune_version == "sp_pig":
    task_kinds = ["prims_single", "prims_on_grid"]
    fd = {"task_kind":task_kinds}            
elif prune_version == "pig_char":
    task_kinds = ["prims_on_grid", "character"]
    fd = {"task_kind":task_kinds}
elif prune_version == "pig_char_0":
    task_kinds = ["prims_on_grid", "character"]
    fd = {"task_kind":task_kinds, "stroke_index":[0]}
elif prune_version == "pig_char_1plus":
    task_kinds = ["prims_on_grid", "character"]
    fd = {"task_kind":task_kinds, "stroke_index":list(range(1, 10))}
else:
    assert False


# (1) Prune to just the desired tasks
SP.DfScalar = SP.DfScalar[SP.DfScalar["task_kind"].isin(task_kinds)].reset_index(drop=True)


In [ ]:
# # (2) Keep only shapes that appear across all task kinds
# TOO SLOW
# from pythonlib.tools.pandastools import extract_with_levels_of_conjunction_vars_helper
# _dfout,_  = extract_with_levels_of_conjunction_vars_helper(SP.DfScalar, "task_kind", [shape_var, "chan"], 
#                                                            n_min_per_lev=n_min_trials_per_shape,
#                                             plot_counts_heatmap_savepath=plot_counts_heatmap_savepath, 
#                                             levels_var=task_kinds)


In [ ]:
df = SP.DfScalar[SP.DfScalar["chan"] == SP.Sites[0]].reset_index(drop=True)

_dfout,_  = extract_with_levels_of_conjunction_vars_helper(df, "task_kind", [shape_var], 
                                                           n_min_per_lev=n_min_trials_per_shape,
                                            plot_counts_heatmap_savepath=plot_counts_heatmap_savepath, 
                                            levels_var=task_kinds)
print(len(df))
print(len(_dfout))
# Get the list of good shapes, and prune SP
shapes_keep = _dfout[shape_var].unique().tolist()
SP.DfScalar = SP.DfScalar[SP.DfScalar["shape_semantic_grp"].isin(shapes_keep)].reset_index(drop=True)

# Plot rasters

In [ ]:
from neuralmonkey.scripts.analy_rasters_script_wrapper import plotter
import os
var = "shape_semantic_grp"
vars_others = ["task_kind", "stroke_index"]
event = "00_stroke"
SAVEDIR = "/tmp/test"
os.makedirs(SAVEDIR, exist_ok=True)
OVERWRITE_n_min = n_min_trials_per_shape
OVERWRITE_lenient_n = 1
plotter(SP, var, vars_others, event, SAVEDIR, OVERWRITE_n_min, OVERWRITE_lenient_n)
